# Notebook 07 — RNA Velocity Analysis (scVelo Dynamical Model)

**Project:** RetinoblastomaAtlas  
**Input:** `data/processed/07_subtyped.h5ad`  
**Output:** `data/processed/08_velocity.h5ad`, velocity stream UMAP, confidence plots, driver genes

---

## Scientific Background

RNA velocity estimates the **future transcriptional state** of each cell by analysing the ratio of unspliced (nascent) to spliced (mature) mRNA. Unspliced pre-mRNA is rapidly produced after gene activation; spliced mRNA accumulates more slowly and decays faster once a gene is silenced. The imbalance between the two forms reveals the **direction and speed** of transcriptional change.

## Significance in This Project

1. **Determine directionality**: Are tumour cells transitioning TOWARD or AWAY FROM the mature cone state? A velocity field pointing toward dedifferentiation supports the Subtype 1 → Subtype 2 invasion model.
2. **Identify transitional states**: Cells at bifurcation points represent critical decision points where TGF-β or macrophage signals may push cells toward the invasive fate.
3. **Estimate kinetic parameters**: Per-gene transcription (α), splicing (β), and degradation (γ) rates identify candidate invasion driver genes.

## Why the Dynamical Model?

The steady-state RNA velocity model (La Manno et al. 2018) assumes all cells are at steady state — violated in tumour evolution where cells are actively transitioning. The **scVelo dynamical model** (Bergen et al. 2020) relaxes this assumption by inferring full transcriptional kinetics per gene via an EM algorithm.

## Input Requirements

RNA velocity requires both **spliced** and **unspliced** count matrices, generated during alignment (STARsolo or Cell Ranger with `--include-introns`). If absent from the h5ad, the pipeline attempts to load them from Velocyto loom files.

**References:**  
- Bergen V et al. *Nat Biotechnol* 2020. https://doi.org/10.1038/s41587-020-0591-3  
- La Manno G et al. *Nature* 2018. https://doi.org/10.1038/s41586-018-0414-6

## Parameters

In [ ]:
N_TOP_GENES       = 2000  # Velocity genes to select
MIN_SHARED_COUNTS = 30    # Min shared counts for cell inclusion
SUBSET_CELL_TYPE  = "Cone_precursor"  # Cell type for subset velocity analysis

## Setup

In [ ]:
import warnings
from pathlib import Path

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1
scv.settings.verbosity = 1
scv.settings.presenter_view = False

ROOT     = Path("..").resolve()
IN_H5AD  = ROOT / "data" / "processed" / "07_subtyped.h5ad"
OUT_H5AD = ROOT / "data" / "processed" / "08_velocity.h5ad"
LOOM_DIR = ROOT / "data" / "raw" / "loom"
FIG_DIR  = ROOT / "results" / "figures"
TAB_DIR  = ROOT / "results" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print(f"scVelo version: {scv.__version__}")

## Step 1 — Load Subtyped Atlas

In [ ]:
print("Step 1 — Loading subtyped atlas")
adata = sc.read_h5ad(IN_H5AD)
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"  Layers: {list(adata.layers.keys())}")

## Step 2 — Check/Load Spliced/Unspliced Layers

RNA velocity requires both `spliced` and `unspliced` count matrices.

**If already present in h5ad**: proceed directly.

**If absent**: load from Velocyto loom files in `data/raw/loom/`. To generate loom files, run Velocyto on the original BAM files:
```
velocyto run10x <cellranger_output_dir> <genome_gtf>
```
or use STARsolo with `--soloFeatures Velocyto`.

In [ ]:
print("Step 2 — Checking for spliced/unspliced layers")

if "spliced" not in adata.layers or "unspliced" not in adata.layers:
    print("  Spliced/unspliced layers not in h5ad. Attempting loom fallback...")
    loom_files = list(LOOM_DIR.glob("*.loom"))
    if not loom_files:
        raise FileNotFoundError(
            f"No .loom files in {LOOM_DIR}.\n"
            "Run: velocyto run10x <cellranger_dir> <genome_gtf>"
        )
    loom_adatas = []
    for loom_path in sorted(loom_files):
        ladata = scv.read(str(loom_path), cache=True)
        ladata.var_names_make_unique()
        loom_adatas.append(ladata)
        print(f"  Loaded loom: {loom_path.name} ({ladata.n_obs:,} cells)")
    loom_combined = loom_adatas[0] if len(loom_adatas) == 1 else sc.concat(loom_adatas)
    adata = scv.utils.merge(adata, loom_combined)
    print(f"  After loom merge: {adata.n_obs:,} cells")
else:
    n_s = (adata.layers["spliced"] > 0).sum()
    n_u = (adata.layers["unspliced"] > 0).sum()
    print(f"  Spliced layer: {n_s:,} non-zero entries")
    print(f"  Unspliced layer: {n_u:,} non-zero entries")

## Step 3 — Preprocessing for Velocity

`scv.pp.filter_and_normalize()` removes cells with fewer than `min_shared_counts` shared spliced/unspliced counts (unreliable ratio due to sampling noise) and selects the top `n_top_genes` velocity genes.

In [ ]:
print("Step 3 — Preprocessing for velocity")
print(f"  Parameters: n_top_genes={N_TOP_GENES}, min_shared_counts={MIN_SHARED_COUNTS}")

scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=MIN_SHARED_COUNTS,
    n_top_genes=N_TOP_GENES,
    log=True,
)
print(f"  After filter_and_normalize: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

## Step 4 — Compute First/Second Order Moments

Moments are computed in the **scVI latent space** (30-d KNN graph) rather than raw PCA. This ensures that velocity neighbours are biologically similar rather than batch-confounded.

Moments smooth the spliced/unspliced counts across nearest neighbours, reducing noise in the velocity estimates.

In [ ]:
print("Step 4 — Computing moments in scVI latent space")

use_rep = "X_scVI" if "X_scVI" in adata.obsm else "X_pca"
print(f"  Using representation: {use_rep}")

scv.pp.moments(
    adata,
    n_pcs=30 if use_rep == "X_pca" else None,
    n_neighbors=30,
    use_rep=use_rep,
)

## Step 5 — Fit Dynamical Model

The dynamical model (Bergen et al. 2020) estimates per-gene:
- **α**: transcription rate
- **β**: splicing rate
- **γ**: degradation rate
- **t_s**: switching time (induction → repression)

This is computationally intensive (~10–60 minutes for 100k cells). Genes with high `fit_likelihood` have coherent spliced/unspliced dynamics and are reliable velocity drivers — candidates for invasion-regulating genes.

In [ ]:
print(f"Step 5 — Fitting dynamical model (n_top_genes={N_TOP_GENES})")
print("  This is compute-intensive. Consider using n_jobs=4+ for speed.")

scv.tl.recover_dynamics(adata, n_jobs=4)
scv.tl.velocity(adata, mode="dynamical")
print("  Dynamical model fitted.")

## Step 6 — Project Velocity onto UMAP

`scv.tl.velocity_graph()` builds a cell-transition probability matrix based on the cosine similarity between each cell's velocity vector and its neighbours' displacement vectors. This graph is used for velocity embedding and pseudotime.

In [ ]:
print("Step 6 — Projecting velocity onto UMAP")

scv.tl.velocity_graph(adata)
scv.tl.velocity_embedding(adata, basis="umap")

# Velocity confidence and pseudotime
scv.tl.velocity_confidence(adata)
scv.tl.velocity_pseudotime(adata)
print("  Velocity confidence and pseudotime computed.")
print("  velocity_pseudotime: 0 = root, 1 = tip of trajectory")

## Step 7 — Velocity Stream Plot

**What to look for:**
- Arrow direction shows where cells are heading transcriptionally
- Divergent arrows at a cluster boundary = bifurcation point (fate decision)
- Convergent arrows toward a cluster = terminal/stable state
- For RB: arrows from early progenitor-like states toward invasive states would support the dedifferentiation model

In [ ]:
color_keys = ["cell_type_broad", "rb_subtype", "velocity_pseudotime"]
color_keys = [k for k in color_keys if k in adata.obs.columns]

ncols = len(color_keys)
fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
if ncols == 1:
    axes = [axes]

for ax, key in zip(axes, color_keys):
    scv.pl.velocity_embedding_stream(
        adata, basis="umap", color=key,
        ax=ax, show=False, frameon=False,
        size=20, alpha=0.7, arrow_size=1.5, linewidth=0.8,
        title=key,
    )

plt.suptitle("RNA velocity stream (scVelo dynamical model)",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / "velocity_stream_umap.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 8 — Velocity Confidence and Length

- **Velocity confidence**: coherence of velocity vectors in local neighbourhood. Low confidence = ambiguous transcriptional state
- **Velocity length**: transcriptional speed. Faster-changing cells are often closer to fate decisions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scv.pl.umap(adata, color="velocity_confidence", ax=axes[0], show=False,
            frameon=False, cmap="RdYlGn", vmin=0, vmax=1,
            title="Velocity confidence")
scv.pl.umap(adata, color="velocity_length", ax=axes[1], show=False,
            frameon=False, cmap="YlOrRd",
            title="Velocity length (transcriptional speed)")
plt.suptitle("RNA velocity quality metrics", fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "velocity_confidence_umap.pdf", dpi=200, bbox_inches="tight")
plt.show()

## Step 9 — Top Driver Genes and Phase Portraits

Phase portraits show spliced vs. unspliced expression. In the dynamical model, the fitted kinetic curve represents the transcriptional cycle. Genes with high `fit_likelihood` and clear portrait structure are the most reliable velocity drivers — candidates for regulating the invasion transition.

In [ ]:
if "fit_likelihood" in adata.var.columns:
    driver_df = (
        adata.var[["fit_likelihood", "fit_alpha", "fit_beta", "fit_gamma",
                   "fit_t_", "gene_count_corr"]]
        .dropna()
        .sort_values("fit_likelihood", ascending=False)
        .head(200)
    )
    driver_df.to_csv(TAB_DIR / "velocity_driver_genes.csv")
    print(f"Top 5 driver genes: {driver_df.head(5).index.tolist()}")
    print(f"Saved driver gene table -> {TAB_DIR / 'velocity_driver_genes.csv'}")

    # Phase portraits of top 12 driver genes
    top_genes = driver_df.head(12).index.tolist()
    ncols = 4
    nrows = int(np.ceil(len(top_genes) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()
    for ax, gene in zip(axes, top_genes):
        scv.pl.velocity(adata, var_names=[gene], ax=ax, show=False)
        ax.set_title(gene, fontsize=9)
    for ax in axes[len(top_genes):]:
        ax.set_visible(False)
    plt.suptitle("Top velocity driver genes (sorted by fit likelihood)",
                 fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "velocity_top_driver_genes.pdf", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("'fit_likelihood' not found — run dynamical model first (Step 5)")

## Step 10 — Cone Precursor Subset Velocity

Restricting to `Cone_precursor` cells provides **high-resolution within-tumour directionality**. In the full atlas, velocity arrows are dominated by immune-to-tumour transitions; the subset view reveals fine trajectory structure within the tumour population.

In [ ]:
if SUBSET_CELL_TYPE in adata.obs["cell_type_broad"].values:
    print(f"Step 10 — Running velocity on {SUBSET_CELL_TYPE} subset")
    adata_sub = adata[adata.obs["cell_type_broad"] == SUBSET_CELL_TYPE].copy()
    print(f"  Subset: {adata_sub.n_obs:,} cells")

    scv.pp.moments(adata_sub, n_pcs=20, n_neighbors=20)
    scv.tl.velocity(adata_sub, mode="dynamical")
    scv.tl.velocity_graph(adata_sub)
    scv.tl.velocity_embedding(adata_sub, basis="umap")
    scv.tl.velocity_pseudotime(adata_sub)

    color_keys = ["cell_type_fine", "velocity_pseudotime", "rb_subtype"]
    color_keys = [k for k in color_keys if k in adata_sub.obs.columns]
    ncols = len(color_keys)
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 6))
    if ncols == 1:
        axes = [axes]
    for ax, key in zip(axes, color_keys):
        scv.pl.velocity_embedding_stream(
            adata_sub, basis="umap", color=key,
            ax=ax, show=False, frameon=False,
            size=20, alpha=0.7, arrow_size=1.5,
            title=key,
        )
    plt.suptitle(f"RNA velocity — {SUBSET_CELL_TYPE} subset",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    fig.savefig(FIG_DIR / f"velocity_stream_{SUBSET_CELL_TYPE.lower()}_subset.pdf",
                dpi=200, bbox_inches="tight")
    plt.show()

    # Store subset pseudotime in full adata
    adata.obs.loc[adata_sub.obs_names, f"velocity_pseudotime_{SUBSET_CELL_TYPE}"] = \
        adata_sub.obs["velocity_pseudotime"]
else:
    print(f"  No '{SUBSET_CELL_TYPE}' cells found — skipping subset velocity")

## Step 11 — Save Velocity-Annotated Atlas

In [ ]:
print(f"Saving velocity-annotated atlas -> {OUT_H5AD.name}")
adata.write_h5ad(OUT_H5AD, compression="gzip")
size_mb = OUT_H5AD.stat().st_size / 1e6
print(f"  {adata.n_obs:,} cells x {adata.n_vars:,} genes | {size_mb:.0f} MB")
print("  Added:")
print("    .layers['velocity']         : velocity vectors")
print("    .obs['velocity_pseudotime'] : RNA velocity pseudotime")
print("    .obs['velocity_confidence'] : per-cell velocity confidence")
print("    .obsm['velocity_umap']      : 2-d velocity embedding")
print("    .var['fit_likelihood']      : per-gene kinetic fit quality")
print("  Next: Run notebook 08_cellrank_fate_mapping.ipynb")